# Phase 2: fine-tune ResNet-50 and ViT-S/16 on KonIQ-10k

Kaggle: GPU T4 or P100, Internet on. Expected: ~30-60 min per run, 6 runs total.
Finished runs are skipped and interrupted runs resume, so you can rerun cells safely.

**Kaggle sessions stop after ~12 h.** If you need to split the work, run one config per session and keep `checkpoints/` + `results/` (download them or save as a Kaggle Dataset).

In [ ]:
!git clone https://github.com/elise-love/Learned-Image-Quality-Assessment-Benchmark.git
%cd Learned-Image-Quality-Assessment-Benchmark
!pip install -q pyiqa timm
!python scripts/download_data.py --datasets koniq10k

In [ ]:
# Colab only: keep checkpoints on Drive so a disconnect does not lose progress
# from google.colab import drive; drive.mount('/content/drive')
# CKPT = '/content/drive/MyDrive/learned-iqa/checkpoints'
CKPT = 'checkpoints'

In [ ]:
# 1. Sanity check: overfit 64 images. Loss should approach 0, SRCC ~0.99. (~3 min)
!python -m src.train --config configs/overfit.yaml --ckpt-dir {CKPT}

In [ ]:
# 2. Quick 1-epoch smoke test of the real config (optional, ~5 min)
!python -m src.train --config configs/resnet50.yaml --seed 99 --ckpt-dir {CKPT}_smoke --set epochs=1 out_dir=results/smoke results_table=results/smoke.csv

In [ ]:
# 3. ResNet-50, 3 seeds
!python -m src.train --config configs/resnet50.yaml --seed 0 1 2 --ckpt-dir {CKPT}

In [ ]:
# 4. ViT-S/16, 3 seeds (if out-of-memory: add  --set batch_size=8)
!python -m src.train --config configs/vit_s16.yaml --seed 0 1 2 --ckpt-dir {CKPT}

In [ ]:
# 5. Tables and figure (also run the phase-1 notebook first so baselines are included)
!python scripts/make_tables.py
!python scripts/plot_scatter.py
from IPython.display import Image, Markdown
display(Markdown(open('results/main_table.md').read()))
Image('results/figures/scatter_koniq.png')

In [ ]:
# 6. Download results/ to put back in your local repo
!zip -qr results.zip results
from IPython.display import FileLink; FileLink('results.zip')